# ACADIA 2023: Building Graph Representation - Training, Validation, and Testing

This notebook demonstrates a complete machine learning workflow for building graph classification using **topologic_fast** and PyTorch Geometric.

## Overview

Building Graph Representation (BGR) is a powerful approach for encoding 3D building geometry as graphs for machine learning applications:

- **Nodes**: Represent spatial units (rooms, cells, zones)
- **Edges**: Represent adjacency relationships (shared walls)
- **Features**: Encode spatial and topological properties

## What You Will Learn

1. **Data Preparation**: Generate building datasets with topologic_fast
2. **Feature Engineering**: Extract meaningful graph features
3. **Model Architecture**: Build effective GNNs for classification
4. **Training Pipeline**: Complete train/validation/test workflow
5. **Evaluation**: Metrics, confusion matrices, and visualization

## Prerequisites

```bash
pip install topologic_fast torch torch_geometric plotly pandas scikit-learn
```

## Import Libraries

In [ ]:
# Core imports
import topologic_fast as tf
import numpy as np
import pandas as pd
from collections import defaultdict
import time
import json

# Deep learning
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import SAGEConv, global_mean_pool, global_max_pool

# Visualization
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ML utilities
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import torch_geometric
print("=" * 60)
print("ACADIA 2023 - Building Graph Representation Workshop")
print("=" * 60)
print(f"topologic_fast: loaded")
print(f"PyTorch: {torch.__version__}")
print(f"PyTorch Geometric: {torch_geometric.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print("=" * 60)

## Part 1: Building Generation

We create diverse building typologies for classification:

- **Type 0: Bar** - Linear, elongated buildings
- **Type 1: Tower** - Compact, tall towers
- **Type 2: Courtyard** - Buildings with interior voids
- **Type 3: L-Shape** - Angular corner buildings
- **Type 4: Podium Tower** - Wide base with narrow tower

In [ ]:
class BuildingGenerator:
    """Generates diverse building topologies for classification."""
    
    TYPES = {0: 'Bar', 1: 'Tower', 2: 'Courtyard', 3: 'L-Shape', 4: 'Podium Tower'}
    
    def __init__(self, floor_height=3.0):
        self.floor_height = floor_height
    
    def generate(self, building_type, **kwargs):
        generators = {
            0: self._bar, 1: self._tower, 2: self._courtyard,
            3: self._l_shape, 4: self._podium_tower
        }
        return generators.get(building_type, self._bar)(**kwargs)
    
    def _bar(self, length=8, width=2, floors=3):
        cells = []
        for i in range(length):
            for j in range(width):
                for f in range(floors):
                    cells.append(tf.Cell.Box(i, j, f * self.floor_height, 1, 1, self.floor_height))
        return tf.CellComplex.ByCells(cells) if cells else None
    
    def _tower(self, base=3, floors=8):
        cells = []
        for i in range(base):
            for j in range(base):
                for f in range(floors):
                    cells.append(tf.Cell.Box(i, j, f * self.floor_height, 1, 1, self.floor_height))
        return tf.CellComplex.ByCells(cells) if cells else None
    
    def _courtyard(self, outer=6, inner=2, floors=4):
        cells = []
        inner_start = (outer - inner) // 2
        inner_end = inner_start + inner
        for i in range(outer):
            for j in range(outer):
                if inner_start <= i < inner_end and inner_start <= j < inner_end:
                    continue
                for f in range(floors):
                    cells.append(tf.Cell.Box(i, j, f * self.floor_height, 1, 1, self.floor_height))
        return tf.CellComplex.ByCells(cells) if cells else None
    
    def _l_shape(self, wing1=5, wing2=5, width=2, floors=4):
        cells = []
        for i in range(wing1):
            for j in range(width):
                for f in range(floors):
                    cells.append(tf.Cell.Box(i, j, f * self.floor_height, 1, 1, self.floor_height))
        for i in range(width):
            for j in range(width, wing2):
                for f in range(floors):
                    cells.append(tf.Cell.Box(i, j, f * self.floor_height, 1, 1, self.floor_height))
        return tf.CellComplex.ByCells(cells) if cells else None
    
    def _podium_tower(self, podium_size=6, tower_size=3, podium_floors=2, tower_floors=6):
        cells = []
        tower_offset = (podium_size - tower_size) // 2
        for i in range(podium_size):
            for j in range(podium_size):
                for f in range(podium_floors):
                    cells.append(tf.Cell.Box(i, j, f * self.floor_height, 1, 1, self.floor_height))
        for i in range(tower_size):
            for j in range(tower_size):
                for f in range(tower_floors):
                    cells.append(tf.Cell.Box(tower_offset + i, tower_offset + j, 
                                             (podium_floors + f) * self.floor_height, 1, 1, self.floor_height))
        return tf.CellComplex.ByCells(cells) if cells else None

# Test generator
generator = BuildingGenerator()
print("Building Types:")
for type_id, name in generator.TYPES.items():
    building = generator.generate(type_id)
    if building:
        graph = tf.Graph.ByTopology(building, direct=True)
        print(f"  {type_id}: {name:15s} - {graph.Order():3d} cells, {graph.Size():3d} adjacencies")

## Part 2: Dataset Generation

Generate a diverse dataset of building graphs with variations for robust training.

In [ ]:
import random

def generate_dataset(samples_per_class=50, seed=42):
    """Generate a diverse dataset of building graphs."""
    random.seed(seed)
    np.random.seed(seed)
    
    generator = BuildingGenerator()
    buildings = []
    labels = []
    
    # Parameter ranges for each building type
    param_ranges = {
        0: {'length': (6, 12), 'width': (2, 3), 'floors': (2, 5)},       # Bar
        1: {'base': (2, 5), 'floors': (5, 12)},                           # Tower
        2: {'outer': (5, 8), 'inner': (1, 3), 'floors': (3, 6)},         # Courtyard
        3: {'wing1': (4, 7), 'wing2': (4, 7), 'width': (2, 3), 'floors': (3, 6)},  # L-Shape
        4: {'podium_size': (5, 8), 'tower_size': (2, 4), 
            'podium_floors': (1, 3), 'tower_floors': (4, 8)}              # Podium Tower
    }
    
    print(f"Generating {samples_per_class} samples per class...")
    
    for building_type in range(5):
        params = param_ranges[building_type]
        count = 0
        attempts = 0
        max_attempts = samples_per_class * 3
        
        while count < samples_per_class and attempts < max_attempts:
            attempts += 1
            # Sample random parameters
            kwargs = {}
            for key, (low, high) in params.items():
                kwargs[key] = random.randint(low, high)
            
            try:
                building = generator.generate(building_type, **kwargs)
                if building:
                    buildings.append(building)
                    labels.append(building_type)
                    count += 1
            except Exception as e:
                continue
        
        print(f"  Type {building_type} ({generator.TYPES[building_type]}): {count} samples")
    
    return buildings, labels

# Generate dataset
buildings, labels = generate_dataset(samples_per_class=40, seed=42)
print(f"\nTotal buildings: {len(buildings)}")
print(f"Label distribution: {dict(zip(*np.unique(labels, return_counts=True)))}")

## Part 3: Graph Conversion

Convert topologic_fast buildings to PyTorch Geometric Data objects.

In [ ]:
def building_to_pyg_data(building, label):
    """Convert a topologic_fast building to PyTorch Geometric Data."""
    # Extract dual graph
    graph = tf.Graph.ByTopology(building, direct=True, tolerance=0.001)
    
    # Get adjacency information
    adj_list = graph.AdjacencyList()
    vertices = graph.Vertices()
    num_nodes = len(vertices)
    
    if num_nodes == 0:
        return None
    
    # Build edge index from adjacency list
    edge_src = []
    edge_dst = []
    
    for src_idx, neighbors in enumerate(adj_list):
        for dst_idx in neighbors:
            edge_src.append(src_idx)
            edge_dst.append(dst_idx)
    
    # Create node features: [x, y, z, degree, normalized_z]
    node_features = []
    max_z = 0
    coords_list = []
    
    for v in vertices:
        coords = v.Coordinates()
        coords_list.append(coords)
        max_z = max(max_z, coords[2])
    
    max_z = max(max_z, 1.0)  # Avoid division by zero
    
    for i, coords in enumerate(coords_list):
        degree = len(adj_list[i]) if i < len(adj_list) else 0
        features = [
            coords[0] / 10.0,           # Normalized x
            coords[1] / 10.0,           # Normalized y
            coords[2] / max_z,          # Normalized z (floor level)
            degree / 6.0,               # Normalized degree
            1.0 if coords[2] == 0 else 0.0,  # Is ground floor
            1.0 if coords[2] == max_z - 3.0 else 0.0  # Is top floor
        ]
        node_features.append(features)
    
    # Create PyG Data object
    x = torch.tensor(node_features, dtype=torch.float)
    edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)
    y = torch.tensor([label], dtype=torch.long)
    
    return Data(x=x, edge_index=edge_index, y=y, num_nodes=num_nodes)

# Convert all buildings to PyG Data
print("Converting buildings to PyG Data objects...")
dataset = []
conversion_stats = {'success': 0, 'failed': 0}

for i, (building, label) in enumerate(zip(buildings, labels)):
    try:
        data = building_to_pyg_data(building, label)
        if data is not None and data.num_nodes > 0 and data.edge_index.shape[1] > 0:
            dataset.append(data)
            conversion_stats['success'] += 1
        else:
            conversion_stats['failed'] += 1
    except Exception as e:
        conversion_stats['failed'] += 1
        if i < 5:  # Print first few errors
            print(f"  Error converting building {i}: {e}")

print(f"\nConversion complete:")
print(f"  Success: {conversion_stats['success']}")
print(f"  Failed: {conversion_stats['failed']}")
print(f"\nDataset size: {len(dataset)} graphs")
print(f"Node features: {dataset[0].x.shape[1] if dataset else 0}")

## Part 4: Train/Validation/Test Split

Split the dataset with stratification to ensure balanced class distribution.

In [ ]:
# Get labels for stratification
dataset_labels = [data.y.item() for data in dataset]

# First split: train+val vs test (85/15)
train_val_data, test_data, train_val_labels, test_labels = train_test_split(
    dataset, dataset_labels, test_size=0.15, stratify=dataset_labels, random_state=42
)

# Second split: train vs val (82/18 of train_val = 70/15 overall)
train_data, val_data, train_labels, val_labels = train_test_split(
    train_val_data, train_val_labels, test_size=0.18, stratify=train_val_labels, random_state=42
)

print("=" * 50)
print("Dataset Split Summary")
print("=" * 50)
print(f"Training:   {len(train_data):3d} graphs ({len(train_data)/len(dataset)*100:.1f}%)")
print(f"Validation: {len(val_data):3d} graphs ({len(val_data)/len(dataset)*100:.1f}%)")
print(f"Testing:    {len(test_data):3d} graphs ({len(test_data)/len(dataset)*100:.1f}%)")
print(f"Total:      {len(dataset):3d} graphs")

# Show class distribution
print("\nClass Distribution:")
for split_name, split_labels in [("Train", train_labels), ("Val", val_labels), ("Test", test_labels)]:
    counts = dict(zip(*np.unique(split_labels, return_counts=True)))
    dist = ", ".join([f"{BuildingGenerator.TYPES[k]}: {v}" for k, v in sorted(counts.items())])
    print(f"  {split_name}: {dist}")

## Part 5: GNN Model Architecture

Define a Graph Neural Network using GraphSAGE convolutions for building classification.

In [ ]:
class BuildingGraphClassifier(nn.Module):
    """
    Graph Neural Network for building typology classification.
    
    Architecture:
    - 3 GraphSAGE convolutional layers with batch normalization
    - Global pooling (mean + max concatenation)
    - 2-layer MLP classifier with dropout
    """
    
    def __init__(self, in_channels, hidden_channels, num_classes, dropout=0.3):
        super().__init__()
        
        # Graph convolution layers
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.bn1 = nn.BatchNorm1d(hidden_channels)
        
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.bn2 = nn.BatchNorm1d(hidden_channels)
        
        self.conv3 = SAGEConv(hidden_channels, hidden_channels)
        self.bn3 = nn.BatchNorm1d(hidden_channels)
        
        # Classifier (2x hidden for mean+max pooling)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_channels * 2, hidden_channels),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_channels, num_classes)
        )
        
        self.dropout = dropout
    
    def forward(self, x, edge_index, batch):
        # Message passing layers
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        
        x = self.conv3(x, edge_index)
        x = self.bn3(x)
        x = F.relu(x)
        
        # Global pooling: concatenate mean and max
        x_mean = global_mean_pool(x, batch)
        x_max = global_max_pool(x, batch)
        x = torch.cat([x_mean, x_max], dim=1)
        
        # Classification
        return self.classifier(x)

# Model configuration
NUM_FEATURES = dataset[0].x.shape[1]  # 6 features
HIDDEN_DIM = 64
NUM_CLASSES = 5
DROPOUT = 0.3

model = BuildingGraphClassifier(
    in_channels=NUM_FEATURES,
    hidden_channels=HIDDEN_DIM,
    num_classes=NUM_CLASSES,
    dropout=DROPOUT
).to(device)

print("Model Architecture:")
print("=" * 50)
print(model)
print("=" * 50)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## Part 6: Training Loop

Train the model with early stopping based on validation loss.

In [ ]:
# Create data loaders
BATCH_SIZE = 16

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)

# Training configuration
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 100
PATIENCE = 15  # Early stopping patience

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

def train_epoch(model, loader, optimizer, criterion):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(out, batch.y)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * batch.num_graphs
        pred = out.argmax(dim=1)
        correct += (pred == batch.y).sum().item()
        total += batch.num_graphs
    
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion):
    """Evaluate model on a dataset."""
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    for batch in loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(out, batch.y)
        
        total_loss += loss.item() * batch.num_graphs
        pred = out.argmax(dim=1)
        correct += (pred == batch.y).sum().item()
        total += batch.num_graphs
        
        all_preds.extend(pred.cpu().numpy())
        all_labels.extend(batch.y.cpu().numpy())
    
    return total_loss / total, correct / total, all_preds, all_labels

# Training loop with early stopping
print("=" * 60)
print("Training Started")
print("=" * 60)

history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_loss = float('inf')
best_model_state = None
patience_counter = 0

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    scheduler.step(val_loss)
    
    # Early stopping check
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict().copy()
        patience_counter = 0
        marker = " *"  # Best model marker
    else:
        patience_counter += 1
        marker = ""
    
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | "
              f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.3f} | "
              f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.3f}{marker}")
    
    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

training_time = time.time() - start_time
print(f"\nTraining completed in {training_time:.1f} seconds")

# Load best model
model.load_state_dict(best_model_state)
print(f"Best validation loss: {best_val_loss:.4f}")

## Part 7: Training Visualization

Visualize training progress with loss and accuracy curves.

In [ ]:
# Plot training history
fig = make_subplots(rows=1, cols=2, subplot_titles=['Loss', 'Accuracy'])

epochs = list(range(1, len(history['train_loss']) + 1))

# Loss plot
fig.add_trace(go.Scatter(x=epochs, y=history['train_loss'], name='Train Loss',
                         line=dict(color='blue')), row=1, col=1)
fig.add_trace(go.Scatter(x=epochs, y=history['val_loss'], name='Val Loss',
                         line=dict(color='red')), row=1, col=1)

# Accuracy plot
fig.add_trace(go.Scatter(x=epochs, y=history['train_acc'], name='Train Acc',
                         line=dict(color='blue')), row=1, col=2)
fig.add_trace(go.Scatter(x=epochs, y=history['val_acc'], name='Val Acc',
                         line=dict(color='red')), row=1, col=2)

fig.update_layout(
    title='Training History',
    height=400,
    width=900,
    showlegend=True
)

fig.update_xaxes(title_text='Epoch', row=1, col=1)
fig.update_xaxes(title_text='Epoch', row=1, col=2)
fig.update_yaxes(title_text='Loss', row=1, col=1)
fig.update_yaxes(title_text='Accuracy', row=1, col=2)

fig.show()

## Part 8: Final Evaluation on Test Set

Evaluate the trained model on the held-out test set.

In [ ]:
# Final evaluation on test set
test_loss, test_acc, test_preds, test_true = evaluate(model, test_loader, criterion)

print("=" * 60)
print("Final Test Results")
print("=" * 60)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f} ({test_acc*100:.1f}%)")

# Classification report
print("\nClassification Report:")
print("-" * 60)
class_names = [BuildingGenerator.TYPES[i] for i in range(5)]
print(classification_report(test_true, test_preds, target_names=class_names, zero_division=0))

# Per-class accuracy
print("\nPer-Class Accuracy:")
for i, name in enumerate(class_names):
    mask = np.array(test_true) == i
    if mask.sum() > 0:
        class_acc = (np.array(test_preds)[mask] == i).mean()
        print(f"  {name:15s}: {class_acc:.3f} ({mask.sum()} samples)")

## Part 9: Confusion Matrix Visualization

Visualize classification performance with an interactive confusion matrix.

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(test_true, test_preds)
cm_normalized = cm.astype('float') / cm.sum(axis=1, keepdims=True)

# Create confusion matrix heatmap
fig = go.Figure(data=go.Heatmap(
    z=cm_normalized,
    x=class_names,
    y=class_names,
    colorscale='Blues',
    text=[[f"{cm[i,j]}\n({cm_normalized[i,j]:.1%})" for j in range(5)] for i in range(5)],
    texttemplate='%{text}',
    textfont=dict(size=12),
    hovertemplate='True: %{y}<br>Predicted: %{x}<br>Count: %{text}<extra></extra>'
))

fig.update_layout(
    title='Confusion Matrix (Test Set)',
    xaxis_title='Predicted Label',
    yaxis_title='True Label',
    width=600,
    height=500,
)

# Reverse y-axis to have (0,0) at top-left
fig.update_yaxes(autorange='reversed')

fig.show()

# Summary statistics
print("\nConfusion Matrix Analysis:")
print(f"  Total correct: {np.trace(cm)} / {cm.sum()}")
print(f"  Most confused: ", end="")
cm_no_diag = cm.copy()
np.fill_diagonal(cm_no_diag, 0)
max_idx = np.unravel_index(cm_no_diag.argmax(), cm_no_diag.shape)
print(f"{class_names[max_idx[0]]} -> {class_names[max_idx[1]]} ({cm_no_diag[max_idx]} times)")

## Summary

This ACADIA 2023 workshop notebook demonstrated a complete Building Graph Representation (BGR) machine learning workflow using **topologic_fast** and PyTorch Geometric.

### Key Components

1. **Building Generation** - Created 5 building typologies (Bar, Tower, Courtyard, L-Shape, Podium Tower) using `tf.Cell.Box()` and `tf.CellComplex.ByCells()`

2. **Graph Extraction** - Converted buildings to dual graphs using `tf.Graph.ByTopology()` where:
   - Nodes = spatial cells (rooms)
   - Edges = adjacency relationships (shared faces)

3. **Feature Engineering** - Extracted 6 node features:
   - Normalized coordinates (x, y, z)
   - Node degree
   - Ground/top floor indicators

4. **Model Architecture** - GraphSAGE-based GNN with:
   - 3 convolutional layers with batch normalization
   - Mean + max global pooling
   - 2-layer MLP classifier

5. **Training Pipeline** - Complete workflow with:
   - Train/validation/test split (70/15/15)
   - Early stopping based on validation loss
   - Learning rate scheduling

6. **Evaluation** - Comprehensive metrics including:
   - Accuracy and loss curves
   - Classification report
   - Interactive confusion matrix

### topologic_fast API Used

```python
# Create cell complex
cells = [tf.Cell.Box(x, y, z, w, l, h) for ...]
cell_complex = tf.CellComplex.ByCells(cells)

# Extract dual graph
graph = tf.Graph.ByTopology(cell_complex, direct=True, tolerance=0.001)

# Get graph structure
adj_list = graph.AdjacencyList()  # List of neighbor indices per node
vertices = graph.Vertices()        # List of vertex objects
num_nodes = graph.Order()          # Number of vertices
num_edges = graph.Size()           # Number of edges
```

### Note on topologicpy Compatibility

The original topologicpy uses wrapper functions like:
- `DGL.DatasetByGraphs()` - Not yet implemented in topologic_fast
- `DGL.Model()` - Not yet implemented in topologic_fast
- `PyG.ByGraph()` - Not yet implemented in topologic_fast

This notebook demonstrates the equivalent workflow using standard PyTorch Geometric directly with topologic_fast's graph extraction capabilities.